# InventoryIQ - GPU Anomaly Detection
## NVIDIA RAPIDS cuML on Colab T4

**Steps:**
1. Make sure runtime is set to GPU (Runtime -> Change runtime type -> T4 GPU)
2. Upload `inventory.csv` when prompted in Cell 2
3. Run all cells
4. Download `anomaly_scores.csv` from the Files panel
5. Put it in your `data/` folder locally and run `python setup_db.py`


In [ ]:
# Cell 1 - Install RAPIDS cuML for Colab CUDA 12
# This takes ~2 minutes on first run. Colab T4 currently uses CUDA 12,
# so avoid cudf/cuML CUDA 11 packages and the cupy-cuda11x conflict.
!pip uninstall -y cudf-cu11 cuml-cu11 cupy-cuda11x -q
!pip install -q --extra-index-url=https://pypi.nvidia.com "cuda-python>=12.0" cupy-cuda12x cuml-cu12
print('RAPIDS cuML installed for CUDA 12')

In [ ]:
# Cell 2 - Upload inventory.csv
from google.colab import files
uploaded = files.upload()  # Upload your data/inventory.csv here
import io
import pandas as pd

filename = list(uploaded.keys())[0]
df_pd = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Loaded {len(df_pd)} SKUs')
print(df_pd.head(3))

In [ ]:
# Cell 3 - Load sales features into GPU memory with CuPy
import cupy as cp

# Select the 12-month sales feature columns
sales_cols = [c for c in df_pd.columns if c.startswith('sales_m')]
print(f'Sales columns: {sales_cols}')

X_np = df_pd[sales_cols].fillna(0).to_numpy(dtype='float32')
X_gpu = cp.asarray(X_np)
print(f'Feature matrix on GPU: {X_gpu.shape}')

In [ ]:
# Cell 4 - RAPIDS cuML Feature Engineering
import cupy as cp

# Add statistical features alongside raw monthly sales
X_np = X_gpu

mean_sales = X_np.mean(axis=1, keepdims=True)
std_sales = X_np.std(axis=1, keepdims=True)
max_sales = X_np.max(axis=1, keepdims=True)
min_sales = X_np.min(axis=1, keepdims=True)
zero_months = (X_np == 0).sum(axis=1, keepdims=True).astype(float)
trend = (X_np[:, 9:] - X_np[:, :3]).mean(axis=1, keepdims=True)  # recent vs early

X_features = cp.hstack([
    X_np,
    mean_sales,
    std_sales,
    max_sales,
    min_sales,
    zero_months,
    trend
])

print(f'Feature matrix with engineered features: {X_features.shape}')

In [ ]:
# Cell 5 - Isolation Forest anomaly detection
# Some current Colab/cuML builds do not expose cuml.ensemble.IsolationForest.
# Fall back to sklearn so the notebook still completes and exports anomaly_scores.csv.
import time
import numpy as np

backend = 'cuML'
start = time.time()

try:
    from cuml.ensemble import IsolationForest as CuMLIsolationForest
    iso = CuMLIsolationForest(
        n_estimators=200,
        contamination=0.15,
        random_state=42,
        verbose=False,
    )
    iso.fit(X_features)
    anomaly_labels = iso.predict(X_features)
    anomaly_scores = iso.score_samples(X_features)
except (ImportError, AttributeError) as e:
    from sklearn.ensemble import IsolationForest as SklearnIsolationForest
    backend = 'sklearn fallback'
    print(f'cuML IsolationForest unavailable: {e}')
    print('Falling back to sklearn IsolationForest for anomaly scores.')
    X_cpu = cp.asnumpy(X_features)
    iso = SklearnIsolationForest(
        n_estimators=200,
        contamination=0.15,
        random_state=42,
    )
    iso.fit(X_cpu)
    anomaly_labels = iso.predict(X_cpu)
    anomaly_scores = iso.score_samples(X_cpu)

elapsed = time.time() - start
anomaly_values = cp.asnumpy(anomaly_labels) if hasattr(anomaly_labels, 'get') else np.asarray(anomaly_labels)
anomaly_count = int((anomaly_values == -1).sum())
print(f'Completed Isolation Forest in {elapsed:.2f}s using {backend}')
print(f'Anomalies detected: {anomaly_count}')

In [ ]:
# Cell 6 - RAPIDS cuML DBSCAN Clustering (velocity clusters)
from cuml.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np

# Use mean + std + zero_months for clustering
X_cluster = cp.hstack([mean_sales, std_sales, zero_months])

# Normalise on CPU for DBSCAN
X_cluster_np = X_cluster.get()
scaler = StandardScaler()
X_cluster_norm = scaler.fit_transform(X_cluster_np)
X_cluster_gpu = cp.asarray(X_cluster_norm.astype('float32'))

dbscan = DBSCAN(eps=0.5, min_samples=5)
cluster_labels = dbscan.fit_predict(X_cluster_gpu)
cluster_labels_np = cp.asnumpy(cluster_labels)

print(f'DBSCAN clusters found: {len(np.unique(cluster_labels_np))} (including noise=-1)')

In [ ]:
# Cell 7 - Build and export results
import pandas as pd
import numpy as np

def to_numpy(x):
    if hasattr(x, 'get'):
        return x.get()
    if hasattr(x, 'to_numpy'):
        return x.to_numpy()
    return np.asarray(x)

results_df = pd.DataFrame({
    'sku_id': df_pd['sku_id'],
    'anomaly_score': to_numpy(anomaly_scores),
    'anomaly_label': to_numpy(anomaly_labels).astype(int),
    'velocity_cluster': to_numpy(cluster_labels).astype(int),
})

# Normalise anomaly score to 0-1 range for readability
s = results_df['anomaly_score']
results_df['anomaly_score_norm'] = (s - s.min()) / (s.max() - s.min())

results_df.to_csv('anomaly_scores.csv', index=False)
print('Saved anomaly_scores.csv')
print(results_df.describe())
print(f"\nAnomalies: {(results_df['anomaly_label'] == -1).sum()}")
print(f"Normal:    {(results_df['anomaly_label'] == 1).sum()}")

In [ ]:
# Cell 8 - Visualise results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score distribution
axes[0].hist(results_df['anomaly_score'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(
    results_df[results_df['anomaly_label'] == -1]['anomaly_score'].max(),
    color='red', linestyle='--', label='Anomaly threshold'
)
axes[0].set_title('Anomaly Score Distribution (Isolation Forest)')
axes[0].set_xlabel('Anomaly Score (more negative = more anomalous)')
axes[0].legend()

# Anomalies vs normals scatter
mean_sales_np = X_features[:, 12].get()  # mean feature
zero_np = X_features[:, 16].get()  # zero_months feature
colors = ['red' if l == -1 else 'steelblue' for l in results_df['anomaly_label']]
axes[1].scatter(mean_sales_np, zero_np, c=colors, alpha=0.5, s=15)
axes[1].set_xlabel('Mean Monthly Sales')
axes[1].set_ylabel('Months with Zero Sales')
axes[1].set_title('SKU Velocity Map (red = anomaly)')

plt.tight_layout()
plt.savefig('anomaly_plot.png', dpi=120)
plt.show()
print('Plot saved as anomaly_plot.png')

In [ ]:
# Cell 9 - Download results
from google.colab import files
files.download('anomaly_scores.csv')
files.download('anomaly_plot.png')
print('Files downloaded!')
print('Next step: put anomaly_scores.csv in your data/ folder and run:')
print('  python setup_db.py')